# Food Calorie AI — نوتبوك التشغيل على Colab
النوتبوك ده بيشغّل كود المشروع اللي في `src/`، مفيش فيه منطق خاص بيه.

قبل ما تبدأ: **Runtime → Change runtime type → T4 GPU**

## 0) الإعداد: نجيب المشروع ونربط Drive

In [ ]:
# غيّر اللينك ده للينك الريبو بتاعك على GitHub
REPO_URL = 'https://github.com/Abdallaallam5/food-calorie-ai.git'

import os
if not os.path.exists('/content/food-calorie-ai'):
    !git clone {REPO_URL} /content/food-calorie-ai
else:
    !cd /content/food-calorie-ai && git pull
%cd /content/food-calorie-ai
!pip install -q -r requirements.txt

In [ ]:
# نحفظ الموديل على Drive عشان لو الجلسة فصلت ما نخسرش التدريب
from google.colab import drive
drive.mount('/content/drive')

os.environ['SAVE_DIR'] = '/content/drive/MyDrive/food_calorie_model'
os.environ['DATA_DIR'] = '/content/data'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'مفيش GPU! غيّر الـ runtime')

## 1) نستكشف الداتا

In [ ]:
import random
import matplotlib.pyplot as plt
from src.data import get_datasets

train_raw, test_raw = get_datasets(with_transforms=False)  # أول مرة بتحمّل ~5GB
print(f'Train: {len(train_raw)} | Test: {len(test_raw)} | Classes: {len(train_raw.classes)}')

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax in axes.flat:
    img, label = train_raw[random.randint(0, len(train_raw) - 1)]
    ax.imshow(img); ax.set_title(train_raw.classes[label]); ax.axis('off')
plt.tight_layout(); plt.show()

## 2) التدريب 🚀
حوالي ساعة لساعة ونص على T4. لو الجلسة فصلت، شغّل خلايا الإعداد تاني وبعدين الخلية دي، وهيكمل من آخر epoch.

In [ ]:
!python -m src.train

## 3) نجرّب على صورة من عندك ونحسب السعرات
- `SIZE = None`: الكمية بتتقدّر من الصورة تلقائي (صوّر الطبق من فوق وهو باين كله).
- أو اختار `'small'` / `'medium'` / `'large'` لو عايز تحدد بنفسك.

In [ ]:
SIZE = None  # None = تقدير من الصورة | 'small' / 'medium' / 'large'

from google.colab import files
from PIL import Image
from src.predict import CalorieEstimator, format_result
from src.portion import visualize

est = CalorieEstimator()
for name in files.upload():
    img = Image.open(name)
    result, portion = est.estimate(img, size=SIZE, return_portion=True)

    fig, axes = plt.subplots(1, 2 if portion else 1, figsize=(12, 6))
    axes = axes if portion else [axes]
    axes[0].imshow(img); axes[0].set_title('الصورة'); axes[0].axis('off')
    if portion:
        axes[1].imshow(visualize(portion)); axes[1].axis('off')
        axes[1].set_title(f'plate (green) / food (orange): {portion.coverage:.0%}')
    plt.show()
    print(format_result(result))
